# UdeA Insurance — Reto Tarifación Seguro de Salud
## Limpieza y organización de los Datos 

### Bases de datos a limpiar y organizar
Cada sección tiene unos pasos a ejecutar, los cuales permiten limpiar y estructurar de manera deseada cada una de las bases de datos de acuerdo al objetivo. La ultima sección tiene los pasos que permitieron llevar a cabo la unifiación y validación de las 3 bases de datos.

Sección 1
- **BD_Exposicion:** Periodos de cobertura de cada asegurado (Contrato)

Sección 2
- **BD_SocioDemografica:** Perfil demográfico y condiciones preexistentes de los afiliados (Asegurados)

Sección 3
- **BD_Siniestros:** Reclamaciones médicas históricas (Asegurados/Afiliados)

Sección 4
- **Integracion** de las 3 bases de datos
1. BD_Exposicion
2. BD_SocioDemografica
3. BD_Siniestros





In [ ]:
# Librerías estándar para realizar limpieza y  análisis de datos
import pandas as pd
import numpy as np
from pathlib import Path

# Visualización
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


In [2]:
#Obtener la raíz del proyecto (subiendo un nivel desde la carpeta src/)
proyecto_root = Path.cwd().parent

# Ruta a la carpeta data
data_dir = proyecto_root / "data"

## Sección 1: Limpieza y organización de BD_Exposicion

### ¿Qué hace esta base de datos?
Es parte del contrato y registra **cuánto tiempo estuvo asegurada cada persona**. 

### Pasos que vamos a ejecutar en esta sección:
1. Diagnóstico inicial (tipos, nulos, duplicados exactos por ID)
2. Conversión de fechas (de texto a datetime)
3. Calcular la **fecha de salida real** (cancelación o fin de contrato)
4. Calcular la **exposición** en días y meses
5. Detectar y reportar inconsistencias si las hay.

# Paso 1: Diagnóstico inicial (tipos, nulos, duplicados exactos por ID)


In [ ]:
# Leer uno de los archivos (df_exposicion) para diagnóstico inicial
df_exposicion = pd.read_csv(data_dir / "BD_Exposicion.txt", sep="\t")  # o sep=',' o sep=';'

# Diagnóstico inicial ──────────────────────────────────────────────────
print('Primeras cinco filas ')
display(df_exposicion.head(5))

print('\n Tipos de datos y nulos por columna')
print(df_exposicion.info())

print('\n Valores nulos por columna y porcentaje')
nulos = df_exposicion.isnull().sum()
pct   = (nulos / len(df_exposicion) * 100).round(2)
print(pd.DataFrame({'nulos': nulos, 'pct_%': pct}))

print(f'\nFilas duplicadas: {df_exposicion.duplicated().sum():,}')
print(f'Asegurados únicos (Asegurado_Id): {df_exposicion["Asegurado_Id"].nunique():,}')
print(f'Pólizas únicas  (Poliza_Asegurado_Id): {df_exposicion["Poliza_Asegurado_Id"].nunique():,}')


Primeras cinco filas 


,Asegurado_Id,Poliza_Asegurado_Id,FECHA_INICIO,FECHA_CANCELACION,FECHA_FIN
0,61630146,271191574,29/01/2024,29/06/2024,29/06/2024
1,7966225,283714899,13/11/2024,NaN,31/12/2024
2,3124492,50801309,1/01/2024,17/01/2024,17/01/2024
3,11863430,236507980,1/01/2024,NaN,31/12/2024
4,4345756,230314288,1/01/2024,NaN,31/12/2024



 Tipos de datos y nulos por columna
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320002 entries, 0 to 320001
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   Asegurado_Id         320002 non-null  int64 
 1   Poliza_Asegurado_Id  320002 non-null  int64 
 2   FECHA_INICIO         320002 non-null  object
 3   FECHA_CANCELACION    109031 non-null  object
 4   FECHA_FIN            320002 non-null  object
dtypes: int64(2), object(3)
memory usage: 12.2+ MB
None

 Valores nulos por columna y porcentaje
                      nulos  pct_%
Asegurado_Id              0   0.00
Poliza_Asegurado_Id       0   0.00
FECHA_INICIO              0   0.00
FECHA_CANCELACION    210971  65.93
FECHA_FIN                 0   0.00

Filas duplicadas: 0
Asegurados únicos (Asegurado_Id): 296,020
Pólizas únicas  (Poliza_Asegurado_Id): 320,002


## IMPORTANTE PRECISIÓN! 
Consultamos literatura para definir conceptos

### ¿Cuál es la diferencia entre "Asegurados únicos" y "Pólizas únicas" en df_exposicion?

- **Asegurados únicos**: Es el número de personas distintas que han estado aseguradas. Se cuenta usando el identificador de persona (`Asegurado_Id`). Si una persona tuvo varias pólizas a lo largo del tiempo, igual cuenta solo una vez como asegurado único.

- **Pólizas únicas**: Es el número de contratos distintos emitidos. Cada póliza es un contrato individual entre la aseguradora y el asegurado, identificado por `Poliza_Asegurado_Id`. Si una persona tuvo varias pólizas (por ejemplo, renovó el seguro o cambió de plan), cada una cuenta como una póliza única, aunque sea el mismo asegurado.

**Ejemplo:**
- Si Juan Pérez tuvo 2 pólizas diferentes en años distintos, cuenta como 1 asegurado único y 2 pólizas únicas.
- Si María Gómez solo tuvo 1 póliza, cuenta como 1 asegurado único y 1 póliza única.

**En resumen:**
- "Asegurados únicos" mide cuántas personas diferentes hay.
- "Pólizas únicas" mide cuántos contratos distintos existen, incluyendo renovaciones o cambios de plan para una misma persona.


> **Nota aclaratoria:**
>
> Si el número de pólizas únicas es mayor que el de asegurados únicos, significa que algunas personas (asegurados únicos) han tenido más de una póliza a lo largo del tiempo. Es decir, hicieron renovaciones, cambios de plan o tuvieron varios contratos distintos. Por eso, el conteo de pólizas supera al de personas distintas.


# Paso 2: Conversión de fechas (de texto a datetime)

In [4]:
# Convertir columnas de fecha antes de operar con ellas
for columna in ['FECHA_INICIO', 'FECHA_FIN', 'FECHA_CANCELACION']:
    df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')

print('Columnas de fecha convertidas a datetime correctamente.')

/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_15871/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')
/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_15871/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')
/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_15871/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')


Columnas de fecha convertidas a datetime correctamente.


## Paso 3: Calcular la **fecha de salida real** (cancelación o fin de contrato)


In [ ]:
# Fecha de salida real ─────────────────────────────────────────────────

df_exposicion['FECHA_SALIDA'] = df_exposicion['FECHA_CANCELACION'].fillna(df_exposicion['FECHA_FIN'])

# Validación extra: si FECHA_CANCELACION ES MAYOR A  FECHA_FIN, hay un error en los datos
# (la cancelación ocurrió DESPUÉS del fin del contrato, lo cual no tiene sentido)
mask_error = (
    df_exposicion['FECHA_CANCELACION'].notna() &
    (df_exposicion['FECHA_CANCELACION'] > df_exposicion['FECHA_FIN'])
)
print(f'Registros con FECHA_CANCELACION > FECHA_FIN (inconsistentes): {mask_error.sum():,}')

# Para esos casos, corregimos usando FECHA_FIN como salida real
df_exposicion.loc[mask_error, 'FECHA_SALIDA'] = df_exposicion.loc[mask_error, 'FECHA_FIN']

print('\n Ejemplo de la nueva columna FECHA_SALIDA:')
.display(df_exposicion[['Asegurado_Id', 'FECHA_INICIO', 'FECHA_CANCELACION', 'FECHA_FIN', 'FECHA_SALIDA']].head(8))

Registros con FECHA_CANCELACION > FECHA_FIN (inconsistentes): 18

 Ejemplo de la nueva columna FECHA_SALIDA:


,Asegurado_Id,FECHA_INICIO,FECHA_CANCELACION,FECHA_FIN,FECHA_SALIDA
0,61630146,2024-01-29,2024-06-29,2024-06-29,2024-06-29
1,7966225,2024-11-13,NaT,2024-12-31,2024-12-31
2,3124492,2024-01-01,2024-01-17,2024-01-17,2024-01-17
3,11863430,2024-01-01,NaT,2024-12-31,2024-12-31
4,4345756,2024-01-01,NaT,2024-12-31,2024-12-31
5,8134114,2024-01-01,NaT,2024-12-31,2024-12-31
6,34285636,2024-01-01,2024-02-18,2024-02-18,2024-02-18
7,54470034,2024-01-01,2024-12-31,2024-12-31,2024-12-31


> **Otra nota aclaratoria jajajaja Todo claro parcero**
>
>FECHA_CANCELACION > FECHA_FIN (inconsistentes), significa que en esos casos la fecha de cancelación del contrato aparece después de la fecha en que el contrato debía terminar. Esto es un error en los datos, porque no tiene sentido que alguien cancele un contrato después de que ya terminó.

# Paso 4: Calcular la **exposición** en días y meses

In [6]:
#Calcular exposición ──────────────────────────────────────────────────

df_exposicion['dias_expuesto'] = (df_exposicion['FECHA_SALIDA'] - df_exposicion['FECHA_INICIO']).dt.days

# Convertir a meses (aproximación: 1 mes = 30.4375 días)
df_exposicion['meses_expuesto'] = df_exposicion['dias_expuesto'] / 30.4375

# Detectar exposiciones inválidas
print(' Diagnóstico de exposición ')
print(f'Exposición negativa (error de fechas):  {(df_exposicion["dias_expuesto"] < 0).sum():,}')
print(f'Exposición = 0 días:                    {(df_exposicion["dias_expuesto"] == 0).sum():,}')
print(f'Exposición > 365 días (más de 1 año):   {(df_exposicion["dias_expuesto"] > 365).sum():,}')
print()
print(df_exposicion['dias_expuesto'].describe())

 Diagnóstico de exposición 
Exposición negativa (error de fechas):  0
Exposición = 0 días:                    6,578
Exposición > 365 días (más de 1 año):   0

count    320002.000000
mean        294.559943
std         116.671127
min           0.000000
25%         244.000000
50%         365.000000
75%         365.000000
max         365.000000
Name: dias_expuesto, dtype: float64


# Paso 5: Detección y reporte de inconsistencias (Si existen)

In [ ]:
# Consolidación final de BD_Exposicion ( para analisis y modelado) ─────────────
# 1) Filtrar exposición válida
n_invalidos = (df_exposicion['dias_expuesto'] <= 0).sum()
pct_invalidos = n_invalidos / len(df_exposicion) * 100

print(f'Exposición <= 0: {n_invalidos:,} ({pct_invalidos:.2f}%)')

df_exposicion['exposicion_valida'] = df_exposicion['dias_expuesto'] > 0
df_exposicion_limpio = df_exposicion[df_exposicion['exposicion_valida']].copy()

# 2) Diagnóstico de duplicados
dup_exactos = df_exposicion_limpio.duplicated().sum()
dup_poliza_fechas = df_exposicion_limpio.duplicated(
    subset=['Poliza_Asegurado_Id', 'FECHA_INICIO', 'FECHA_SALIDA']
).sum()
dup_poliza = df_exposicion_limpio.duplicated(subset=['Poliza_Asegurado_Id']).sum()

print('\n=== Resumen final de BD_Exposicion ===')
print(f'Duplicados exactos: {dup_exactos:,}')
print(f'Duplicados por póliza+fechas: {dup_poliza_fechas:,}')
print(f'Pólizas repetidas (cualquier periodo): {dup_poliza:,}')

# 3) Base final de modelado (remueve solo duplicado exacto)
df_exposicion_modelo = df_exposicion_limpio.drop_duplicates().copy()

print('\n=== información final para modelado ===')
print(f'Filas finales: {len(df_exposicion_modelo):,}')
print(f'Asegurados únicos: {df_exposicion_modelo["Asegurado_Id"].nunique():,}')
print(f'Pólizas únicas: {df_exposicion_modelo["Poliza_Asegurado_Id"].nunique():,}')

Exposición <= 0: 6,578 (2.06%)

=== Resumen final de BD_Exposicion ===
Duplicados exactos: 0
Duplicados por póliza+fechas: 0
Pólizas repetidas (cualquier periodo): 0

=== información final para modelado ===
Filas finales: 313,424
Asegurados únicos: 293,288
Pólizas únicas: 313,424


**IMPORTANTE!** Hago la siguiente observación, si le restamos a Pólizas únicas: 313,424 los Asegurados únicos: 293,288 esa diferencia son los Asegurados que tienen mas de una poliza.

## Sección 2: Limpieza y organización de BD_SocioDemografica

### ¿Qué hace esta base de datos?
Contiene el **perfil de riesgo** de cada asegurado (UNA fila por Afiliado_Id). Aquí están las variables que más influyen en el costo de un seguro de salud: edad, sexo, ciudad y enfermedades preexistentes.

### Sección  de 4 pasos:
- **Paso 1**: Diagnóstico inicial (nulos, tipos, duplicados)
- **Paso 2**: Convertir FechaNacimiento a datetime y corregir años futuros
- **Paso 3**: Calcular edad y crear grupos de edad
- **Paso 4**: Crear variables derivadas (num_condiciones, Sexo_Cd_limpio, CIUDAD_NORM) Y Validación y cierre final  df_sociodemograficos_limpio listo para EDA)


# Paso 1: Diagnóstico Inicial
Nulos, tipos, duplicados por Afiliado_Id

In [ ]:
# Leer la base de sociodemográficos ( El codigo ajusta el separador si hace falta)
df_sociodemograficos = pd.read_csv(data_dir / "BD_SocioDemografica.txt", sep="\t", encoding="latin1")

# -Diagnóstico inicial ──────────────────────────────────────
print('Primeras filas de la bd_sociodemograficos:')
display(df_sociodemograficos.head())

print('\n Informacion general ')
df_sociodemograficos.info()

print('\n  Valores únicos en Sexo_Cd ')
print(df_sociodemograficos['Sexo_Cd'].value_counts())

print('\nDuplicados por Afiliado_Id ')
print(f'Duplicados encontrados: {df_sociodemograficos.duplicated(subset=["Afiliado_Id"]).sum()}')



Primeras filas de la bd_sociodemograficos:


,Afiliado_Id,Sexo_Cd,FechaNacimiento,CIUDAD,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,ENF_PULMONAR
0,921437,F,30/04/68,Medellin,0,0,0,0,0
1,60504878,M,18/02/12,Medellin,0,0,0,0,0
2,55074222,F,23/10/14,Medellin,0,0,0,0,0
3,23690690,F,27/06/89,Cartagena,0,0,0,0,0
4,45506882,M,30/06/09,Cali,0,0,0,0,0



 Informacion general 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261267 entries, 0 to 261266
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Afiliado_Id      261267 non-null  int64 
 1   Sexo_Cd          261267 non-null  object
 2   FechaNacimiento  261267 non-null  object
 3   CIUDAD           261267 non-null  object
 4   CANCER           261267 non-null  int64 
 5   DIABETES         261267 non-null  int64 
 6   ENF_CARDIACA     261267 non-null  int64 
 7   HIPERTENSION     261267 non-null  int64 
 8   ENF_PULMONAR     261267 non-null  int64 
dtypes: int64(6), object(3)
memory usage: 17.9+ MB

  Valores únicos en Sexo_Cd 
Sexo_Cd
F     140329
M     120902
-1        36
Name: count, dtype: int64

Duplicados por Afiliado_Id 
Duplicados encontrados: 2


In [9]:
# Identificar duplicados exactos por Afiliado_Id y eliminar
n_antes = len(df_sociodemograficos)
print(display(df_sociodemograficos[df_sociodemograficos.duplicated(subset=['Afiliado_Id'], keep=False)]))

,Afiliado_Id,Sexo_Cd,FechaNacimiento,CIUDAD,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,ENF_PULMONAR
32990,62942960,F,24/03/25,Cartagena,0,0,0,0,0
80214,58987622,F,10/10/14,Medellin,0,0,0,0,0
165000,62942960,F,24/03/25,Bogota,0,0,0,0,0
240541,58987622,F,10/10/14,Medellin,0,0,0,0,0


None


In [10]:
# Eliminación de Duplicados Exactos
df_sociodemograficos = df_sociodemograficos.drop_duplicates(subset=['Afiliado_Id'], keep='first')

n_despues = len(df_sociodemograficos)
n_eliminados = n_antes - n_despues

print('ELIMINACIÓN DE DUPLICADOS EXACTOS')
print(f'Filas antes:    {n_antes:,}')
print(f'Filas después:  {n_despues:,}')
print(f'Eliminadas:     {n_eliminados}')


ELIMINACIÓN DE DUPLICADOS EXACTOS
Filas antes:    261,267
Filas después:  261,265
Eliminadas:     2


# Paso 2: Convertir Fechas
Convertir FechaNacimiento a datetime y corregir años futuros (parseo de 2 dígitos)

In [11]:
# Convertir FechaNacimiento a datetime 
df_sociodemograficos['FechaNacimiento_dt'] = pd.to_datetime(
    df_sociodemograficos['FechaNacimiento'],
    dayfirst=True,
    errors='coerce'
)

print('Fechas convertidas correctamente.')
print(f"Fechas no parseadas (NaT): {df_sociodemograficos['FechaNacimiento_dt'].isna().sum():,}")

/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_15871/651047863.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_sociodemograficos['FechaNacimiento_dt'] = pd.to_datetime(


Fechas convertidas correctamente.
Fechas no parseadas (NaT): 0


# Paso 3: Calcular Edad y Grupos de Edad

**Edad**: Años cumplidos al 2026-04-01

**grupo_edad**: Binning en 6 categorías (0-17, 18-30, 31-45, 46-60, 61-75, 76+)

**num_condiciones**: Suma de 5 enfermedades preexistentes (CANCER, DIABETES, ENF_CARDIACA, HIPERTENSION, ENF_PULMONAR)  
- 0 = ninguna enfermedad
- 1-5 = número de enfermedades presentes

In [12]:
#Calcular edad desde FechaNacimiento_dt
FECHA_CORTE = pd.Timestamp('2026-04-01')

# Ajuste para años futuros por parseo de fechas con 2 dígitos
mask_futuro = df_sociodemograficos['FechaNacimiento_dt'] > FECHA_CORTE
df_sociodemograficos.loc[mask_futuro, 'FechaNacimiento_dt'] = (
    df_sociodemograficos.loc[mask_futuro, 'FechaNacimiento_dt'] - pd.DateOffset(years=100)
)

# Edad entera: con redondeo hacia abajo en años cumplidos
df_sociodemograficos['edad'] = np.floor(
    (FECHA_CORTE - df_sociodemograficos['FechaNacimiento_dt']).dt.days / 365.25
).astype('Int64')

print('Edad calculada correctamente.')
print(f"Edades nulas: {df_sociodemograficos['edad'].isna().sum():,}")
print(f"Edades < 0: {(df_sociodemograficos['edad'] < 0).sum():,}")
print(f"Edades > 100: {(df_sociodemograficos['edad'] > 100).sum():,}")

Edad calculada correctamente.
Edades nulas: 0
Edades < 0: 0
Edades > 100: 0


In [ ]:
#Grupos de edad (bins) para análisis exploratorio
bins = [0, 17, 30, 45, 60, 75, 120]
labels = ['0-17', '18-30', '31-45', '46-60', '61-75', '76+']

df_sociodemograficos["grupo_edad"] = pd.cut(
    df_sociodemograficos["edad"],
    bins=[0, 17, 30, 45, 60, 75, 120],
    labels=["0-17", "18-30", "31-45", "46-60", "61-75", "76+"],
    right=True,
    include_lowest=True
)

# Número de condiciones preexistentes
cols_binarias = ['CANCER', 'DIABETES', 'ENF_CARDIACA', 'HIPERTENSION', 'ENF_PULMONAR']
df_sociodemograficos['num_condiciones'] = df_sociodemograficos[cols_binarias].sum(axis=1)

print('Variables derivadas creadas')
print(df_sociodemograficos['grupo_edad'].value_counts(dropna=False).sort_index())
print('\nDistribución de num_condiciones:')
print(df_sociodemograficos['num_condiciones'].value_counts().sort_index())



Variables derivadas creadas
grupo_edad
0-17     57915
18-30    38036
31-45    86612
46-60    50767
61-75    23163
76+       4772
Name: count, dtype: int64

Distribución de num_condiciones:
num_condiciones
0    226859
1     27804
2      5391
3      1054
4       144
5        13
Name: count, dtype: int64


# Paso 4: Crear Variables Derivadas
Normalizar CIUDAD (mayúsculas, sin acentos), Codificar Sexo_Cd_limpio (F|M|DESCONOCIDO u NOBINARIO) Honestamente, nos apoyamos de la literatura para encontrarle significado a ese -1, pero habian diversas interpretaciones por tanto, decidimos renombrarlo (`NOBINARIO`)

In [14]:
# Validación y Cierre Final

import unicodedata

print('\nNORMALIZACION VARIABLE CIUDAD')
if 'CIUDAD_NORM' not in df_sociodemograficos.columns:
    def normalizar_texto(texto):
        if pd.isna(texto):
            return texto
        texto = str(texto).strip().upper()
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
        return texto

    df_sociodemograficos['CIUDAD_NORM'] = df_sociodemograficos['CIUDAD'].apply(normalizar_texto)

print('Ejemplo de normalizacion de CIUDAD:')
print(df_sociodemograficos[['CIUDAD', 'CIUDAD_NORM']].drop_duplicates().head(10))

print('\nCODIFICAR VARIABLES CATEGORICAS (SEXO):')
df_sociodemograficos['Sexo_Cd_limpio'] = (
    df_sociodemograficos['Sexo_Cd'].astype(str).str.upper().where(
        df_sociodemograficos['Sexo_Cd'].astype(str).str.upper().isin(['F', 'M']),
        'NOBINARIO'
    )
)

print('Valores Sexo_Cd_limpio:')
print(df_sociodemograficos['Sexo_Cd_limpio'].value_counts().to_string())

# Consolidacion final con columnas acordadas
columnas_finales = [
    'Afiliado_Id',
    'FechaNacimiento_dt',
    'edad',
    'grupo_edad',
    'CIUDAD_NORM',
    'Sexo_Cd_limpio',
    'CANCER',
    'DIABETES',
    'ENF_CARDIACA',
    'HIPERTENSION',
    'ENF_PULMONAR',
    'num_condiciones'
]

df_sociodemograficos_limpio = df_sociodemograficos[columnas_finales].copy()

print('\nBase final creada: df_sociodemograficos_limpio')


NORMALIZACION VARIABLE CIUDAD
Ejemplo de normalizacion de CIUDAD:
               CIUDAD      CIUDAD_NORM
0            Medellin         MEDELLIN
3           Cartagena        CARTAGENA
4                Cali             CALI
6              Bogota           BOGOTA
2525  Sin Informacion  SIN INFORMACION

CODIFICAR VARIABLES CATEGORICAS (SEXO):
Valores Sexo_Cd_limpio:
Sexo_Cd_limpio
F            140327
M            120902
NOBINARIO        36

Base final creada: df_sociodemograficos_limpio


In [15]:
df_sociodemograficos_limpio.info()

<class 'pandas.core.frame.DataFrame'>
Index: 261265 entries, 0 to 261266
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Afiliado_Id         261265 non-null  int64         
 1   FechaNacimiento_dt  261265 non-null  datetime64[ns]
 2   edad                261265 non-null  Int64         
 3   grupo_edad          261265 non-null  category      
 4   CIUDAD_NORM         261265 non-null  object        
 5   Sexo_Cd_limpio      261265 non-null  object        
 6   CANCER              261265 non-null  int64         
 7   DIABETES            261265 non-null  int64         
 8   ENF_CARDIACA        261265 non-null  int64         
 9   HIPERTENSION        261265 non-null  int64         
 10  ENF_PULMONAR        261265 non-null  int64         
 11  num_condiciones     261265 non-null  int64         
dtypes: Int64(1), category(1), datetime64[ns](1), int64(7), object(2)
memory usage: 24.4+ MB


## Sección 3: Limpieza y organización de BD_Siniestros

### ¿Qué hace esta base de datos?
Contiene cada **reclamación médica individual**: quién reclamó, cuándo, qué diagnóstico
tenía y cuánto costó. Es la base del modelo de predicción de costos.

### Sección compuesta  de 6 pasos a ejecutar:
- Paso 1: Diagnóstico inicial
- Paso 2: Convertir fecha de reclamación
- Paso 3: Analizar y limpiar Valor_Pagado
- Paso 4: Dummizacion de tipo de reclamacion
- Paso 5: Agrupar por Afiliado_Id

# Paso 1: Diagnóstico inicial 
Información general de la base de datos, tipo de datos, datos nulos, datos duplicados etc..

In [16]:
# Leer la base de siniestros (ajusta el separador si hace falta)
df_siniestros = pd.read_csv(data_dir / "BD_Siniestros.txt", sep="\t", encoding="latin1")

# Diagnóstico inicial ──────────────────────────────────────────────────
print('=== Primeras filas ===')
display(df_siniestros.head(5))

print('\n=== Info ===')
print(df_siniestros.info())

print('\n=== Resumen de nulos y faltantes ===')
columnas_texto = df_siniestros.select_dtypes(include=['object', 'string']).columns
columnas_num = df_siniestros.columns.difference(columnas_texto)

nulos_sin = df_siniestros.isna().sum()
pct_sin = (nulos_sin / len(df_siniestros) * 100).round(3)

faltantes_texto = df_siniestros[columnas_texto].apply(
    lambda s: s.astype('string').str.strip().eq('').sum()
)

resumen = pd.DataFrame({
    'dtype': df_siniestros.dtypes.astype(str),
    'nulos': nulos_sin,
    'pct_nulos': pct_sin,
    'faltantes_texto': 0
})
resumen.loc[columnas_texto, 'faltantes_texto'] = faltantes_texto
resumen['faltantes_totales'] = resumen['nulos'] + resumen['faltantes_texto']

print('--- Columnas numéricas / fecha ---')
display(resumen.loc[columnas_num].sort_values('nulos', ascending=False))

print('--- Columnas de texto ---')
display(resumen.loc[columnas_texto].sort_values('faltantes_totales', ascending=False))

=== Primeras filas ===


,Fecha_Reclamacion,Afiliado_Id,Reclamacion,Diagnostico_Codigo,Diagnostico_Desc,Eventos,Valor_Pagado
0,4/12/2024,31586355,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,101576.02545
1,15/01/2024,22753623,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,80630.89152
2,27/05/2024,29321930,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,90237.30633
3,6/11/2024,5258725,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,90237.30633
4,19/07/2024,22815238,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,8346.55713



=== Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1919949 entries, 0 to 1919948
Data columns (total 7 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Fecha_Reclamacion   object 
 1   Afiliado_Id         int64  
 2   Reclamacion         object 
 3   Diagnostico_Codigo  object 
 4   Diagnostico_Desc    object 
 5   Eventos             int64  
 6   Valor_Pagado        float64
dtypes: float64(1), int64(2), object(4)
memory usage: 102.5+ MB
None

=== Resumen de nulos y faltantes ===
--- Columnas numéricas / fecha ---


,dtype,nulos,pct_nulos,faltantes_texto,faltantes_totales
Afiliado_Id,int64,0,0.0,0,0
Eventos,int64,0,0.0,0,0
Valor_Pagado,float64,0,0.0,0,0


--- Columnas de texto ---


,dtype,nulos,pct_nulos,faltantes_texto,faltantes_totales
Fecha_Reclamacion,object,0,0.0,0,0
Reclamacion,object,0,0.0,0,0
Diagnostico_Codigo,object,0,0.0,0,0
Diagnostico_Desc,object,0,0.0,0,0


# Paso 2: Convertir Fecha_Reclamación 

In [17]:
df_siniestros['Fecha_Reclamacion'] = pd.to_datetime(
    df_siniestros['Fecha_Reclamacion'], dayfirst=True, errors='coerce'
)

print('Rango de fechas de reclamaciones:')
print(f'  Mínima: {df_siniestros["Fecha_Reclamacion"].min()}')
print(f'  Máxima: {df_siniestros["Fecha_Reclamacion"].max()}')
print(f'  NaT (no parseadas): {df_siniestros["Fecha_Reclamacion"].isna().sum():,}')

Rango de fechas de reclamaciones:
  Mínima: 2024-01-01 00:00:00
  Máxima: 2024-12-31 00:00:00
  NaT (no parseadas): 0


Codigo para validar si existen datos duplicados exactos

In [18]:
dup_exactos = df_siniestros.duplicated().sum()

llave_negocio = ['Afiliado_Id', 'Fecha_Reclamacion', 'Reclamacion',
'Diagnostico_Codigo', 'Valor_Pagado']
dup_negocio = df_siniestros.duplicated(subset=llave_negocio).sum()

print('Duplicados exactos:', dup_exactos)
print('Duplicados de negocio:', dup_negocio)

Duplicados exactos: 0
Duplicados de negocio: 0


Estandarizacion  de variables texto

In [19]:
for c in ['Reclamacion', 'Diagnostico_Codigo', 'Diagnostico_Desc']:
    df_siniestros[c] = (
df_siniestros[c].astype(str).str.strip().str.upper()
)

Revisar por variable texto (Objeto) cantidad de categorias unicas.

In [20]:
cols = ['Reclamacion', 'Diagnostico_Codigo', 'Diagnostico_Desc']

for c in cols:
    print('\n===', c, '===')
    print('Categorías únicas:', df_siniestros[c].nunique(dropna=False))
    print(df_siniestros[c].value_counts(dropna=False).head(20))


=== Reclamacion ===
Categorías únicas: 8
Reclamacion
CONSULTA MEDICA                1212864
EXAMENES MEDICOS                570237
CIRUGIA                          77008
LABORATORIO                      28543
HOSPITALIZACIONES SIMPLES        13339
HOSPITALIZACIONES COMPLEJAS       8557
CONSULTAS                         6714
TRATAMIENTO DE CANCER             2687
Name: count, dtype: int64

=== Diagnostico_Codigo ===
Categorías únicas: 4372
Diagnostico_Codigo
9       1585021
Z017      87697
Z108      38210
Z014      10799
Z000       9988
I10X       7846
R688       7777
Z321       5485
R529       5292
Z008       5101
L82X       3770
D239       3657
Z019       3460
B07X       3458
Z018       2916
L570       2770
Z011       1922
N390       1863
Z123       1862
D229       1839
Name: count, dtype: int64

=== Diagnostico_Desc ===
Categorías únicas: 4333
Diagnostico_Desc
DIAGNÓSTICO PENDIENTE                                                             1585025
EXAMEN DE LABORATORIO             

## Paso 3: Analizar y limpiar Valor_Pagado

### Definición conceptual de las variables: Valor_Pagado y Eventos

- Analizamos la variable `Valor_Pagado` (el monto pagado por la aseguradora en cada reclamación) y la variable `Eventos` (el número de atenciones, procedimientos o servicios médicos asociados a cada reclamación).

**Significado de "Eventos"**
- Un "evento" es una unidad de atención médica asociada a la reclamación. Puede ser una consulta, procedimiento, cirugía, hospitalización, etc. Por ejemplo, si una reclamación incluye 3 procedimientos médicos, entonces `Eventos = 3`.


In [21]:
print("=== Validación de Valor_Pagado y Eventos ===")

print("\n--- Valor_Pagado ---")
print(df_siniestros["Valor_Pagado"].describe(percentiles=[.5, .75, .9, .95, .99]))
print("Valores negativos:", (df_siniestros["Valor_Pagado"] < 0).sum())
print("Valores iguales a 0:", (df_siniestros["Valor_Pagado"] == 0).sum())

print("\n--- Eventos ---")
print(df_siniestros["Eventos"].describe(percentiles=[.5, .75, .9, .95, .99]))
print("Valores menores o iguales a 0:", (df_siniestros["Eventos"] <= 0).sum())

=== Validación de Valor_Pagado y Eventos ===

--- Valor_Pagado ---
count    1.919949e+06
mean     6.593518e+05
std      5.644707e+06
min      1.574822e+00
50%      1.105525e+05
75%      2.278768e+05
90%      8.571026e+05
95%      1.730479e+06
99%      1.017384e+07
max      1.330849e+09
Name: Valor_Pagado, dtype: float64
Valores negativos: 0
Valores iguales a 0: 0

--- Eventos ---
count    1.919949e+06
mean     1.151845e+00
std      1.284459e+00
min      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      2.000000e+00
99%      3.000000e+00
max      6.300000e+01
Name: Eventos, dtype: float64
Valores menores o iguales a 0: 0



**Interpretación de los resultados**

Segun los resultados obtenidos en la validación en la mayoría de los casos, las reclamaciones tienen un solo evento (por eso la mediana y los percentiles bajos son 1), pero hay reclamaciones complejas que agrupan varios eventos (el máximo observado es 63). Se pude inferir que hay datos atipicos.

- `Valor_Pagado`:
    - No hay valores negativos ni en cero, lo cual es correcto.
    - Hay mucha dispersión: la mayoría de pagos son bajos, pero existen algunos muy altos (ver el máximo y percentiles altos).
- `Eventos`:
    - El valor mínimo es 1 (no hay reclamaciones sin eventos, lo cual es correcto).
    - El promedio es 1.15, la mediana es 1, y el 99% de las reclamaciones tienen hasta 3 eventos.
    - Solo unas pocas reclamaciones agrupan muchos eventos (máximo 63), lo que puede indicar casos complejos o agrupaciones administrativas.

### Justificación del porque NO eliminamos atípicos en esta etapa

Para llevar a cabo posteriormente  el análisis exploratorio de datos (EDA), **no sería  recomendable eliminar los valores atípicos (outliers) de inmediato**. Aquí nuestras  razones profe:

- **Información valiosa:** Algunos atípicos corresponden a casos extremos válidos (por ejemplo, siniestros de alto costo) que pueden ser  relevantes para el negocio y el modelado de riesgos.
- **Decisión informada:** La decisión de eliminar, transformar o conservar atípicos debe tomarse después de entender su origen y su impacto en el análisis o modelo.

Por ahora, solo identificamos y posteriormente los analizaremos.
Tomaremos una decisión sobre su tratamiento en la etapa de preparación final para modelado, cuando tengamos más contexto y claridad sobre su impacto.

## Paso 4: Dummizacion de tipo de reclamacion
Convertimos `Reclamacion` a columnas `rec_*` a nivel de fila de siniestro.


En este paso, vamos a transformar la variable categórica `Reclamacion` en varias columnas binarias (dummies), una por cada tipo de reclamación. Así, cada fila de siniestro tendrá indicadores (0 o 1) que muestran si corresponde a cada tipo de reclamación. Esto nos facilitará el análisis posteriormente.

In [22]:
# Dummies por tipo de reclamacion (a nivel de fila de siniestro)
dummies_reclamacion = pd.get_dummies(df_siniestros['Reclamacion'], prefix='rec', dtype=int)

# Agregar dummies por afiliado (conteos)
df_reclamacion_agg = (
    pd.concat([df_siniestros[['Afiliado_Id']], dummies_reclamacion], axis=1)
    .groupby('Afiliado_Id', as_index=False)
    .sum()
)

print('Dummizacion lista: df_reclamacion_agg')
print(f'Filas: {len(df_reclamacion_agg):,}')
print(f'Tipos de reclamacion convertidos a columnas: {len([c for c in df_reclamacion_agg.columns if c.startswith("rec_")])}')
display(df_reclamacion_agg.head(10))

Dummizacion lista: df_reclamacion_agg
Filas: 240,108
Tipos de reclamacion convertidos a columnas: 8


,Afiliado_Id,rec_CIRUGIA,rec_CONSULTA MEDICA,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER
0,738294,0,4,0,4,0,0,0,0
1,738303,0,1,0,6,0,0,0,0
2,738305,0,6,0,1,0,0,0,0
3,738310,1,5,0,3,0,0,0,0
4,738318,0,0,0,3,0,0,0,0
5,738321,2,4,0,6,0,0,0,0
6,738348,0,5,0,4,0,0,0,0
7,738349,0,1,2,1,0,1,0,0
8,738358,0,11,0,1,0,0,0,0
9,738364,3,7,0,2,0,0,0,0


# Paso 5: Agrupar por Afiliado_Id
En este paso, vamos a resumir la información de siniestros a nivel de cada afiliado. Es decir, agrupamos todas las reclamaciones de un mismo afiliado para obtener variables agregadas como el total de eventos, el monto total pagado, el número de reclamaciones y las fechas de primera y última reclamación. Esto permite analizar y modelar los datos a nivel de persona asegurada.

In [ ]:
# Agregado minimo por Afiliado_Id e integracion de dummies

df_siniestros_agg = df_siniestros.groupby('Afiliado_Id', as_index=False).agg(
    n_reclamaciones=('Reclamacion', 'size'),
    n_eventos=('Eventos', 'sum'),
    total_pagado=('Valor_Pagado', 'sum'),
    fecha_primera_reclamacion=('Fecha_Reclamacion', 'min'),
    fecha_ultima_reclamacion=('Fecha_Reclamacion', 'max')
)

# Unir dummies de reclamacion agregadas por afiliado
df_siniestros_agg = df_siniestros_agg.merge(df_reclamacion_agg, on='Afiliado_Id', how='left')

print('Base agregada: df_siniestros_agg')
print(f'Filas finales: {len(df_siniestros_agg):,}')
print(f'Asegurados unicos: {df_siniestros_agg["Afiliado_Id"].nunique():,}')
print(f'Columnas de tipo reclamacion agregadas: {len([c for c in df_siniestros_agg.columns if c.startswith("rec_")])}')
display(df_siniestros_agg.head(10))

Base agregada: df_siniestros_agg
Filas finales: 240,108
Asegurados unicos: 240,108
Columnas de tipo reclamacion agregadas: 8


,Afiliado_Id,n_reclamaciones,n_eventos,total_pagado,fecha_primera_reclamacion,fecha_ultima_reclamacion,rec_CIRUGIA,rec_CONSULTA MEDICA,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER
0,738294,8,10,3.131529e+06,2024-02-13,2024-07-17,0,4,0,4,0,0,0,0
1,738303,7,8,4.135152e+06,2024-02-09,2024-12-23,0,1,0,6,0,0,0,0
2,738305,7,8,8.020569e+05,2024-02-19,2024-11-16,0,6,0,1,0,0,0,0
3,738310,9,9,2.292458e+06,2024-11-20,2024-12-23,1,5,0,3,0,0,0,0
4,738318,3,3,3.929181e+05,2024-04-20,2024-07-15,0,0,0,3,0,0,0,0
5,738321,12,13,1.822294e+07,2024-01-17,2024-11-06,2,4,0,6,0,0,0,0
6,738348,9,9,3.267979e+06,2024-01-04,2024-10-30,0,5,0,4,0,0,0,0
7,738349,5,5,2.843624e+07,2024-02-12,2024-04-06,0,1,2,1,0,1,0,0
8,738358,12,13,1.821912e+06,2024-02-15,2024-11-28,0,11,0,1,0,0,0,0
9,738364,12,12,4.627797e+06,2024-03-01,2024-12-12,3,7,0,2,0,0,0,0


## Sección 4: Integracion de las 3 bases de datos 

Unificamos exposición, sociodemográficos y siniestros en una sola base a nivel de afiliado y validamos calidad de la unión.

## Pasos a ejecutar en esta sección

- Paso 1: Analisis individual de presencia de ID en cada base de datos
- Paso 2: Imputación de datos nulos en base de datos unificada
- Paso 3: Verificación si la imputación de los datos en la base de datos que `df_modelo_para_tarifación` fue correcta
- Paso 4: Generacion de base de datos unificada para EDA (Analisis exploratorio de datos)

In [24]:
# 1) Preparar exposición a nivel afiliado

df_exposicion_merge = df_exposicion_modelo.copy()
if 'Afiliado_Id' not in df_exposicion_merge.columns and 'Asegurado_Id' in df_exposicion_merge.columns:
    df_exposicion_merge = df_exposicion_merge.rename(columns={'Asegurado_Id': 'Afiliado_Id'})

print('Columnas de llave en exposicion:', [c for c in ['Afiliado_Id', 'Asegurado_Id'] if c in df_exposicion_merge.columns])

df_exposicion_agg = df_exposicion_merge.groupby('Afiliado_Id', as_index=False).agg(
    dias_expuesto_total=('dias_expuesto', 'sum'),
    meses_expuesto_total=('meses_expuesto', 'sum'),
    n_polizas=('Poliza_Asegurado_Id', 'nunique')
)

# 2) Asegurar unicidad en sociodemográficos

df_socio_agg = df_sociodemograficos_limpio.drop_duplicates(subset=['Afiliado_Id']).copy()

# 3) Unión final (base maestra: sociodemográficos)

df_modelo_base = (
    df_socio_agg
    .merge(df_exposicion_agg, on='Afiliado_Id', how='left')
    .merge(df_siniestros_agg, on='Afiliado_Id', how='left')
)

# 4) Validaciones de calidad de la unión

print('=== Base final unificada: df_modelo_base ===')
print(f'Filas: {len(df_modelo_base):,}')
print(f'Afiliados unicos: {df_modelo_base["Afiliado_Id"].nunique():,}')
print(f'Duplicados por Afiliado_Id: {df_modelo_base.duplicated(subset=["Afiliado_Id"]).sum():,}')

resumen_nulos_final = pd.DataFrame({
    'nulos': df_modelo_base.isna().sum(),
    'pct_%': (df_modelo_base.isna().mean() * 100).round(2)
}).sort_values('pct_%', ascending=False)

print('\nTop 15 columnas con mayor porcentaje de nulos:')
display(resumen_nulos_final.head(15))

print('\nVista rápida de la base final:')
display(df_modelo_base.head(10))

Columnas de llave en exposicion: ['Afiliado_Id']
=== Base final unificada: df_modelo_base ===
Filas: 261,265
Afiliados unicos: 261,265
Duplicados por Afiliado_Id: 0

Top 15 columnas con mayor porcentaje de nulos:


,nulos,pct_%
rec_EXAMENES MEDICOS,41453,15.87
rec_LABORATORIO,41453,15.87
rec_HOSPITALIZACIONES SIMPLES,41453,15.87
rec_HOSPITALIZACIONES COMPLEJAS,41453,15.87
rec_CIRUGIA,41453,15.87
rec_CONSULTA MEDICA,41453,15.87
rec_CONSULTAS,41453,15.87
fecha_ultima_reclamacion,41453,15.87
n_eventos,41453,15.87
total_pagado,41453,15.87



Vista rápida de la base final:


,Afiliado_Id,FechaNacimiento_dt,edad,grupo_edad,CIUDAD_NORM,Sexo_Cd_limpio,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,...,fecha_primera_reclamacion,fecha_ultima_reclamacion,rec_CIRUGIA,rec_CONSULTA MEDICA,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER
0,921437,1968-04-30,57,46-60,MEDELLIN,F,0,0,0,0,...,2024-03-04,2024-07-10,0.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0
1,60504878,2012-02-18,14,0-17,MEDELLIN,M,0,0,0,0,...,2024-02-12,2024-09-03,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,55074222,2014-10-23,11,0-17,MEDELLIN,F,0,0,0,0,...,2024-04-05,2024-12-04,2.0,17.0,0.0,4.0,1.0,1.0,0.0,0.0
3,23690690,1989-06-27,36,31-45,CARTAGENA,F,0,0,0,0,...,2024-02-27,2024-05-16,0.0,3.0,0.0,4.0,0.0,0.0,0.0,0.0
4,45506882,2009-06-30,16,0-17,CALI,M,0,0,0,0,...,2024-03-18,2024-07-15,0.0,3.0,0.0,1.0,0.0,1.0,0.0,0.0
5,22954185,1992-03-19,34,31-45,CALI,M,0,0,0,0,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7990436,1965-12-15,60,46-60,BOGOTA,F,0,0,1,1,...,2024-01-04,2024-11-26,0.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0
7,63438236,2012-06-23,13,0-17,CALI,M,0,0,0,0,...,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,42848880,1997-05-31,28,18-30,CARTAGENA,F,0,0,0,0,...,2024-01-10,2024-05-31,0.0,2.0,0.0,2.0,1.0,0.0,0.0,0.0
9,8099835,1977-07-21,48,46-60,MEDELLIN,M,0,0,0,0,...,2024-03-22,2024-07-05,1.0,7.0,0.0,1.0,0.0,0.0,0.0,0.0


## Paso 1: Analisis individual de presencia de ID en cada base de datos y eliminación de inconsistencias.

En este paso, se analiza la presencia de cada ID de afiliado en las diferentes bases de datos (exposición, sociodemográficos y siniestros) para identificar coincidencias y diferencias. Esto permite detectar usuarios que solo aparecen en una o dos fuentes y posibles inconsistencias. Además, se crean banderas de trazabilidad y se eliminan registros inconsistentes para asegurar que la base final esté alineada y lista para el análisis.


In [ ]:
print('=== DIAGNÓSTICO: ¿En qué bases está cada usuario? ===\n')

# Crear banderas de presencia
usuarios_en_socio = set(df_socio_agg['Afiliado_Id'])
usuarios_en_exposicion = set(df_exposicion_agg['Afiliado_Id'])
usuarios_en_siniestros = set(df_siniestros_agg['Afiliado_Id'])

print(f'Total usuarios en sociodemográficos: {len(usuarios_en_socio):,}')
print(f'Total usuarios en exposición: {len(usuarios_en_exposicion):,}')
print(f'Total usuarios en siniestros: {len(usuarios_en_siniestros):,}')

# Intersecciones
socio_exposi = len(usuarios_en_socio & usuarios_en_exposicion)
socio_siniestros = len(usuarios_en_socio & usuarios_en_siniestros)
exposi_siniestros = len(usuarios_en_exposicion & usuarios_en_siniestros)
todos_tres = len(usuarios_en_socio & usuarios_en_exposicion & usuarios_en_siniestros)

print(f'\nSocio + Exposición: {socio_exposi:,}')
print(f'Socio + Siniestros: {socio_siniestros:,}')
print(f'Socio + Exposición + Siniestros: {todos_tres:,}')

# Solo en socio
solo_socio = len(usuarios_en_socio - usuarios_en_exposicion - usuarios_en_siniestros)
print(f'\nSolo en sociodemográficos (sin exposición ni siniestros): {solo_socio:,}')

# En socio y exposición pero sin siniestros
socio_exposi_sin_siniestros = len((usuarios_en_socio & usuarios_en_exposicion) - usuarios_en_siniestros)
print(f'En socio + exposición, pero SIN siniestros: {socio_exposi_sin_siniestros:,}')

# En socio y siniestros pero sin exposición
socio_siniestros_sin_exposi = len((usuarios_en_socio & usuarios_en_siniestros) - usuarios_en_exposicion)
print(f'En socio + siniestros, pero SIN exposición: {socio_siniestros_sin_exposi:,}')

# En socio pero sin ninguna de las otras
sin_exposi_sin_siniestros = len(usuarios_en_socio - usuarios_en_exposicion - usuarios_en_siniestros)
print(f'En socio pero sin NINGUNA de las otras: {sin_exposi_sin_siniestros:,}')

# Crear columnas de trazabilidad en el modelo base
df_modelo_base['en_exposicion'] = df_modelo_base['Afiliado_Id'].isin(usuarios_en_exposicion).astype(int)
df_modelo_base['en_siniestros'] = df_modelo_base['Afiliado_Id'].isin(usuarios_en_siniestros).astype(int)

# Categoría
def categorizar(row):
    if row['en_exposicion'] == 1 and row['en_siniestros'] == 1:
        return 'Completo (Exp + Sin)'
    elif row['en_exposicion'] == 1 and row['en_siniestros'] == 0:
        return 'Solo Exposición'
    elif row['en_exposicion'] == 0 and row['en_siniestros'] == 1:
        return 'Solo Siniestros'
    else:
        return 'Solo Sociodemográficos'

df_modelo_base['categoria_cobertura'] = df_modelo_base.apply(categorizar, axis=1)

print('\n=== Distribución por categoría ===')
print(df_modelo_base['categoria_cobertura'].value_counts())

print('\nVista de muestra con categorías:')
display(df_modelo_base[['Afiliado_Id', 'en_exposicion', 'en_siniestros', 'categoria_cobertura']].head(20))

=== DIAGNÓSTICO: ¿En qué bases está cada usuario? ===

Total usuarios en sociodemográficos: 261,265
Total usuarios en exposición: 293,288
Total usuarios en siniestros: 240,108

Socio + Exposición: 260,853
Socio + Siniestros: 219,812
Socio + Exposición + Siniestros: 219,800

Solo en sociodemográficos (sin exposición ni siniestros): 400
En socio + exposición, pero SIN siniestros: 41,053
En socio + siniestros, pero SIN exposición: 12
En socio pero sin NINGUNA de las otras: 400

=== Distribución por categoría ===
categoria_cobertura
Completo (Exp + Sin)      219800
Solo Exposición            41053
Solo Sociodemográficos       400
Solo Siniestros               12
Name: count, dtype: int64

Vista de muestra con categorías:


,Afiliado_Id,en_exposicion,en_siniestros,categoria_cobertura
0,921437,1,1,Completo (Exp + Sin)
1,60504878,1,1,Completo (Exp + Sin)
2,55074222,1,1,Completo (Exp + Sin)
3,23690690,1,1,Completo (Exp + Sin)
4,45506882,1,1,Completo (Exp + Sin)
5,22954185,1,0,Solo Exposición
6,7990436,1,1,Completo (Exp + Sin)
7,63438236,1,0,Solo Exposición
8,42848880,1,1,Completo (Exp + Sin)
9,8099835,1,1,Completo (Exp + Sin)


In [26]:
# 219,800 que tienen exposición registrada
df_modelo_para_tarifacion = df_modelo_base[df_modelo_base['en_exposicion'] == 1].copy()

print(f'Base original: {len(df_modelo_base):,}')
print(f'Base para tarifación: {len(df_modelo_para_tarifacion):,}')
print(f'Eliminados: {len(df_modelo_base) - len(df_modelo_para_tarifacion):,}')

Base original: 261,265
Base para tarifación: 260,853
Eliminados: 412


# Paso 2: Imputación de datos nulos en base de datos unificada

En este paso, identificamos y reemplazamos los valores nulos (faltantes) en la base de datos unificada. Esto es fundamental para asegurar que los análisis y modelos posteriores no se vean afectados por datos incompletos. La imputación se realiza de forma lógica según el tipo de variable: por ejemplo, se rellenan con 0 los conteos o montos de siniestros cuando no hay información, y se ajustan los tipos de datos para mantener la coherencia.

In [27]:
# Imputacion de variables de siniestros (conteos y monto) con 0
# Aplica a df_modelo_para_tarifacion (219,800 registros con exposición)

cols_base_siniestros = ['n_reclamaciones', 'n_eventos', 'total_pagado']
cols_rec = [c for c in df_modelo_para_tarifacion.columns if c.startswith('rec_')]
cols_fill_zero = cols_base_siniestros + cols_rec

# Rellenar nulos con 0 en variables de frecuencia/monto de siniestros
df_modelo_para_tarifacion[cols_fill_zero] = df_modelo_para_tarifacion[cols_fill_zero].fillna(0)

# Ajustar tipo para columnas de conteo
cols_conteo = ['n_reclamaciones', 'n_eventos'] + cols_rec
df_modelo_para_tarifacion[cols_conteo] = df_modelo_para_tarifacion[cols_conteo].astype(int)

print('Imputacion con 0 aplicada en variables de siniestros (base para tarifación).')
print(f'Columnas imputadas: {len(cols_fill_zero)}')
print('Nulos restantes en esas columnas:', int(df_modelo_para_tarifacion[cols_fill_zero].isna().sum().sum()))
display(df_modelo_para_tarifacion[cols_fill_zero].head(10))

Imputacion con 0 aplicada en variables de siniestros (base para tarifación).
Columnas imputadas: 11
Nulos restantes en esas columnas: 0


,n_reclamaciones,n_eventos,total_pagado,rec_CIRUGIA,rec_CONSULTA MEDICA,rec_CONSULTAS,rec_EXAMENES MEDICOS,rec_HOSPITALIZACIONES COMPLEJAS,rec_HOSPITALIZACIONES SIMPLES,rec_LABORATORIO,rec_TRATAMIENTO DE CANCER
0,4,4,1.183321e+06,0,3,0,1,0,0,0,0
1,2,2,7.389831e+06,0,1,0,0,0,1,0,0
2,25,26,2.670483e+07,2,17,0,4,1,1,0,0
3,7,7,2.339913e+06,0,3,0,4,0,0,0,0
4,5,5,4.718329e+06,0,3,0,1,0,1,0,0
5,0,0,0.000000e+00,0,0,0,0,0,0,0,0
6,4,5,2.787934e+06,0,3,0,1,0,0,0,0
7,0,0,0.000000e+00,0,0,0,0,0,0,0,0
8,5,5,1.050986e+07,0,2,0,2,1,0,0,0
9,9,9,1.995059e+07,1,7,0,1,0,0,0,0


In [ ]:
# Imputacion de variables de exposicion con trazabilidad
# Aplica a df_modelo_para_tarifacion (219,800 registros con exposición)

cols_exposicion = ['meses_expuesto_total', 'n_polizas']

# Bandera para identificar afiliados sin match en exposicion
# (util para EDA y para explicar decisiones de imputacion)
df_modelo_para_tarifacion['sin_exposicion_match'] = df_modelo_para_tarifacion[cols_exposicion].isna().any(axis=1).astype(int)

print('Afiliados sin match en exposicion (en base de tarifación):', int(df_modelo_para_tarifacion['sin_exposicion_match'].sum()))

# Imputacion conservadora: si no hay exposicion en el cruce, asumir 0
df_modelo_para_tarifacion[cols_exposicion] = df_modelo_para_tarifacion[cols_exposicion].fillna(0)

# Tipos esperados
df_modelo_para_tarifacion['n_polizas'] = df_modelo_para_tarifacion['n_polizas'].astype(int)

print('Imputacion de exposicion aplicada a base para tarifación.')
print('Nulos restantes en exposicion:', int(df_modelo_para_tarifacion[cols_exposicion].isna().sum().sum()))
print(f'\nBase final para tarifación: {len(df_modelo_para_tarifacion):,} registros')
print('Variables listas para análisis exploratorio y modelado.')
display(df_modelo_para_tarifacion[cols_exposicion + ['sin_exposicion_match']].head(10))

Afiliados sin match en exposicion (en base de tarifación): 0
Imputacion de exposicion aplicada a base para tarifación.
Nulos restantes en exposicion: 0

Base final para tarifación: 260,853 registros
Variables listas para análisis exploratorio y modelado.


,meses_expuesto_total,n_polizas,sin_exposicion_match
0,11.991786,1,0
1,11.991786,1,0
2,11.991786,1,0
3,11.991786,1,0
4,11.991786,1,0
5,11.991786,1,0
6,11.991786,1,0
7,4.533881,1,0
8,11.991786,1,0
9,11.991786,1,0


Variables listas para análisis exploratorio y modelado.


## Paso 3: Verificación si la imputación de los datos en la base de datos que `df_modelo_para_tarifación` fue correcta

In [29]:
resumen_nulos_final = pd.DataFrame({
    'nulos': df_modelo_para_tarifacion.isna().sum(),
    'pct_%': (df_modelo_para_tarifacion.isna().mean() * 100).round(2)
}).sort_values('pct_%', ascending=False)

print('\nTop 15 columnas con mayor porcentaje de nulos:')
display(resumen_nulos_final.head(15))


Top 15 columnas con mayor porcentaje de nulos:


,nulos,pct_%
fecha_primera_reclamacion,41053,15.74
fecha_ultima_reclamacion,41053,15.74
Afiliado_Id,0,0.00
FechaNacimiento_dt,0,0.00
CIUDAD_NORM,0,0.00
Sexo_Cd_limpio,0,0.00
edad,0,0.00
grupo_edad,0,0.00
ENF_CARDIACA,0,0.00
HIPERTENSION,0,0.00


## Paso 4:  Generacion de base de datos unificada para EDA (Analisis exploratorio de datos)

La base de datos generada se guarda en la carpeta data de `PROYECTO_ANALITICA_II` Luego se comenta el codigo para que no se repita mas bases de datos.

In [30]:
#from pathlib import Path

# Esto apunta a /PROYECTO_ANALITICA_II
#proyecto_root = Path.cwd().parent
#data_dir = proyecto_root / "data"
#data_dir.mkdir(parents=True, exist_ok=True)

#ruta_salida = data_dir / "df_modelo_para_tarifacion.csv"
#df_modelo_para_tarifacion.to_csv(ruta_salida, index=False, encoding="utf-8")

#print("Archivo guardado en:", ruta_salida)